# Extracción de Datos

In [4]:
from pathlib import Path
import pandas as pd

RUTA_CSV = Path("../../siniestralidad_2016_2024.csv")

if not RUTA_CSV.exists():
    raise FileNotFoundError(f"No se encontró el archivo fuente: {RUTA_CSV.resolve()}")

df_raw = pd.read_csv(
    RUTA_CSV,
    sep=";",
    encoding="latin-1",
    dtype="string",
    na_values=[".", ""],
    low_memory=False
)

print(f"Registros cargados: {len(df_raw):,}")
print("Columnas:")
print(df_raw.columns.tolist())

display(df_raw.head())
display(df_raw.isna().sum())

Registros cargados: 371,003
Columnas:
['Código', 'Fecha', 'Tipo confirmado', 'Dirección reporte', 'Tipo clase de accidente', 'Tipo clase de accidente.1', 'Tipo confirmado.1', 'Medio de reporte', 'Numero vehículos', 'Tipo de vehículos implicados']


,Código,Fecha,Tipo confirmado,Dirección reporte,Tipo clase de accidente,Tipo clase de accidente.1,Tipo confirmado.1,Medio de reporte,Numero vehículos,Tipo de vehículos implicados
0,96159,1/01/2016,Solo daños,Carrera 56 Con Calle 1 A,<NA>,<NA>,Solo Da?os,RADIO,2,MOTOCICLETA
1,96160,1/01/2016,Negativo,Calle 72 O Con Carrera 28 H,<NA>,<NA>,Negativo,RADIO,<NA>,<NA>
2,96161,1/01/2016,Con lesionado,Carrera 94 Con Calle 4 C,ATROPELLO,ATROPELLO,Por Atropello,TELEF?NICO 127,2,MOTOCICLETA
3,96162,1/01/2016,Con lesionado,Carrera 8 A Con Calle 44,VOLCAMIENTO,VOLCAMIENTO,Por Volcamiento,RADIO,1,MOTOCICLETA
4,96163,1/01/2016,Negativo,Carrera 28 H Con Calle 72 M,<NA>,<NA>,Negativo,TELEF?NICO 127,<NA>,<NA>


Código                               1
Fecha                                1
Tipo confirmado                      3
Dirección reporte               183686
Tipo clase de accidente         325258
Tipo clase de accidente.1       325258
Tipo confirmado.1                23389
Medio de reporte                288099
Numero vehículos                194424
Tipo de vehículos implicados    222214
dtype: int64

# Transformación del DataFrame

In [5]:
df = df_raw.copy()

df = df.rename(columns={
    "Código": "id_siniestro",
    "Fecha": "fecha",
    "Tipo confirmado": "tipo_confirmado_1",
    "Dirección reporte": "direccion_reporte",
    "Tipo clase de accidente": "tipo_accidente_1",
    "Tipo clase de accidente.1": "tipo_accidente_2",
    "Tipo confirmado.1": "tipo_confirmado_2",
    "Medio de reporte": "medio_reporte",
    "Numero vehículos": "numero_vehiculos",
    "Tipo de vehículos implicados": "tipos_vehiculos"
})

# Unifica las columnas duplicadas usando el primer valor disponible
df["tipo_accidente_raw"] = df["tipo_accidente_1"].combine_first(
    df["tipo_accidente_2"]
)

df["tipo_confirmado_raw"] = df["tipo_confirmado_1"].combine_first(
    df["tipo_confirmado_2"]
)

# Elimina las columnas duplicadas originales
df = df.drop(columns=[
    "tipo_accidente_1",
    "tipo_accidente_2",
    "tipo_confirmado_1",
    "tipo_confirmado_2"
])

# Conversión de tipos
df["id_siniestro"] = df["id_siniestro"].astype("string")
df["fecha"] = pd.to_datetime(
    df["fecha"],
    format="%d/%m/%Y",
    errors="coerce"
)
df["numero_vehiculos"] = pd.to_numeric(
    df["numero_vehiculos"],
    errors="coerce"
).astype("Int64")

print(df.dtypes)
display(df.head())

id_siniestro                   string
fecha                  datetime64[us]
direccion_reporte              string
medio_reporte                  string
numero_vehiculos                Int64
tipos_vehiculos                string
tipo_accidente_raw             string
tipo_confirmado_raw            string
dtype: object


,id_siniestro,fecha,direccion_reporte,medio_reporte,numero_vehiculos,tipos_vehiculos,tipo_accidente_raw,tipo_confirmado_raw
0,96159,2016-01-01,Carrera 56 Con Calle 1 A,RADIO,2,MOTOCICLETA,<NA>,Solo daños
1,96160,2016-01-01,Calle 72 O Con Carrera 28 H,RADIO,<NA>,<NA>,<NA>,Negativo
2,96161,2016-01-01,Carrera 94 Con Calle 4 C,TELEF?NICO 127,2,MOTOCICLETA,ATROPELLO,Con lesionado
3,96162,2016-01-01,Carrera 8 A Con Calle 44,RADIO,1,MOTOCICLETA,VOLCAMIENTO,Con lesionado
4,96163,2016-01-01,Carrera 28 H Con Calle 72 M,TELEF?NICO 127,<NA>,<NA>,<NA>,Negativo


# Control de Duplicados y Claves

In [6]:
# Eliminar filas completamente idénticas
df = df.drop_duplicates().copy()

# Eliminar registros sin identificador de siniestro
df = df.dropna(subset=["id_siniestro"]).copy()

# Conservar un solo registro por siniestro, priorizando el más completo
df["_completitud"] = df.notna().sum(axis=1)

df = (
    df.sort_values("_completitud", ascending=False)
    .drop_duplicates("id_siniestro", keep="first")
    .drop(columns="_completitud")
    .copy()
)

print(f"Registros después de la limpieza: {len(df):,}")
print(
    "Códigos duplicados:",
    df["id_siniestro"].duplicated().sum()
)
print(
    "Códigos nulos:",
    df["id_siniestro"].isna().sum()
)

Registros después de la limpieza: 370,998
Códigos duplicados: 0
Códigos nulos: 0


# Normalización de Textos

In [7]:
columnas_texto = [
    "direccion_reporte",
    "tipo_accidente_raw",
    "tipo_confirmado_raw",
    "medio_reporte",
    "tipos_vehiculos"
]

for columna in columnas_texto:
    df[columna] = (
        df[columna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )

display(df.head())

,id_siniestro,fecha,direccion_reporte,medio_reporte,numero_vehiculos,tipos_vehiculos,tipo_accidente_raw,tipo_confirmado_raw
370971,AI013516,2024-11-02,CALLE 36 & CARRERA 47,AGENTE RADIO,2,"AUTOMOVIL,MOTOCICLETA",CHOQUE,CON LESIONADO
370950,AI013495,2024-11-01,CALLE 33 & CARRERA 98B,CAT 127,2,"MOTOCICLETA,BUSETA",CHOQUE,CON LESIONADO
370967,AI013512,2024-11-02,CALLE 6 & CARRERA 27,AGENTE RADIO,3,"MOTOCICLETA,AUTOMOVIL,AUTOMOVIL",CHOQUE,NEGATIVO
370968,AI013513,2024-11-02,CALLE 16 & CARRERA 83,AGENTE RADIO,2,"AUTOMOVIL,MICROBUS",CHOQUE,CON LESIONADO
370941,AI013486,2024-11-01,CALLE 70 & CARRERA 5,CAT 127,1,MOTOCICLETA,CHOQUE,NEGATIVO


# Estandarizar las Categorías

In [8]:
import numpy as np

texto_gravedad = df["tipo_confirmado_raw"].fillna("")
texto_accidente = df["tipo_accidente_raw"].fillna("")
texto_medio = df["medio_reporte"].fillna("")

# Estandarización de la gravedad
df["gravedad_estandarizada"] = np.select(
    [
        texto_gravedad.str.contains("FALLEC", na=False),
        texto_gravedad.str.contains("LESION|HERID", na=False),
        texto_gravedad.str.contains("SOLO DA", na=False),
        texto_gravedad.str.contains("NEGATIV", na=False),
        df["tipo_confirmado_raw"].isna()
    ],
    [
        "CON FALLECIDOS",
        "CON HERIDOS",
        "SOLO DANOS",
        "NEGATIVO",
        "SIN DATO"
    ],
    default="OTRO"
)

# Estandarización del tipo de accidente
df["tipo_accidente_estandarizado"] = np.select(
    [
        texto_accidente.str.contains("ATROPEL", na=False),
        texto_accidente.str.contains("VOLCAM", na=False),
        texto_accidente.str.contains("CHOQUE", na=False),
        texto_accidente.str.contains("CA.?DA.*OCUPANTE", na=False),
        texto_accidente.str.contains("INCEND", na=False),
        df["tipo_accidente_raw"].isna()
    ],
    [
        "ATROPELLO",
        "VOLCAMIENTO",
        "CHOQUE",
        "CAIDA DE OCUPANTE",
        "INCENDIO",
        "SIN DATO"
    ],
    default="OTRO"
)

# Estandarización del medio de reporte
df["medio_reporte_estandarizado"] = np.select(
    [
        texto_medio.str.contains("123", na=False),
        texto_medio.str.contains("127", na=False),
        texto_medio.str.contains("RADIO", na=False),
        texto_medio.str.contains("AGENTE", na=False),
        texto_medio.str.contains("RED", na=False),
        df["medio_reporte"].isna()
    ],
    [
        "LINEA 123",
        "LINEA 127",
        "RADIO",
        "AGENTE DE TRANSITO",
        "REDES SOCIALES",
        "SIN DATO"
    ],
    default="OTRO-NARRATIVA"
)

display(df[
    [
        "tipo_confirmado_raw",
        "gravedad_estandarizada",
        "tipo_accidente_raw",
        "tipo_accidente_estandarizado",
        "medio_reporte",
        "medio_reporte_estandarizado"
    ]
].head(10))

,tipo_confirmado_raw,gravedad_estandarizada,tipo_accidente_raw,tipo_accidente_estandarizado,medio_reporte,medio_reporte_estandarizado
370971,CON LESIONADO,CON HERIDOS,CHOQUE,CHOQUE,AGENTE RADIO,RADIO
370950,CON LESIONADO,CON HERIDOS,CHOQUE,CHOQUE,CAT 127,LINEA 127
370967,NEGATIVO,NEGATIVO,CHOQUE,CHOQUE,AGENTE RADIO,RADIO
370968,CON LESIONADO,CON HERIDOS,CHOQUE,CHOQUE,AGENTE RADIO,RADIO
370941,NEGATIVO,NEGATIVO,CHOQUE,CHOQUE,CAT 127,LINEA 127
370957,CON LESIONADO,CON HERIDOS,ATROPELLO,ATROPELLO,AGENTE RADIO,RADIO
370947,CON LESIONADO,CON HERIDOS,ATROPELLO,ATROPELLO,AGENTE CAD 123,LINEA 123
370948,NEGATIVO,NEGATIVO,CHOQUE,CHOQUE,AGENTE RADIO,RADIO
370956,CON LESIONADO,CON HERIDOS,CHOQUE,CHOQUE,CAT 127,LINEA 127
370945,NEGATIVO,NEGATIVO,CHOQUE,CHOQUE,CAT 127,LINEA 127


# Dimensión de Tiempo

In [9]:
dim_tiempo = (
    df[["fecha"]]
    .dropna()
    .drop_duplicates()
    .sort_values("fecha")
    .reset_index(drop=True)
)

dim_tiempo["id_tiempo"] = (
    dim_tiempo["fecha"].dt.strftime("%Y%m%d").astype(int)
)

dim_tiempo["anio"] = dim_tiempo["fecha"].dt.year
dim_tiempo["trimestre"] = dim_tiempo["fecha"].dt.quarter
dim_tiempo["mes"] = dim_tiempo["fecha"].dt.month
dim_tiempo["nombre_mes"] = dim_tiempo["fecha"].dt.month_name()
dim_tiempo["dia"] = dim_tiempo["fecha"].dt.day
dim_tiempo["dia_semana_num"] = dim_tiempo["fecha"].dt.dayofweek + 1
dim_tiempo["nombre_dia_semana"] = dim_tiempo["fecha"].dt.day_name()
dim_tiempo["es_fin_de_semana"] = (
    dim_tiempo["fecha"].dt.dayofweek >= 5
)
dim_tiempo["semana_anio"] = (
    dim_tiempo["fecha"].dt.isocalendar().week.astype(int)
)

dim_tiempo = dim_tiempo[
    [
        "id_tiempo",
        "fecha",
        "anio",
        "trimestre",
        "mes",
        "nombre_mes",
        "dia",
        "dia_semana_num",
        "nombre_dia_semana",
        "es_fin_de_semana",
        "semana_anio"
    ]
]

display(dim_tiempo.head())
print(f"Registros en DIM_TIEMPO: {len(dim_tiempo):,}")

,id_tiempo,fecha,anio,trimestre,mes,nombre_mes,dia,dia_semana_num,nombre_dia_semana,es_fin_de_semana,semana_anio
0,20160101,2016-01-01,2016,1,1,January,1,5,Friday,False,53
1,20160102,2016-01-02,2016,1,1,January,2,6,Saturday,True,53
2,20160103,2016-01-03,2016,1,1,January,3,7,Sunday,True,53
3,20160104,2016-01-04,2016,1,1,January,4,1,Monday,False,1
4,20160105,2016-01-05,2016,1,1,January,5,2,Tuesday,False,1


Registros en DIM_TIEMPO: 3,229


# Dimensión de Gravedad

In [10]:
dim_gravedad = (
    df[
        [
            "tipo_confirmado_raw",
            "gravedad_estandarizada"
        ]
    ]
    .drop_duplicates("gravedad_estandarizada")
    .sort_values("gravedad_estandarizada")
    .reset_index(drop=True)
)

dim_gravedad.insert(
    0,
    "id_gravedad",
    range(1, len(dim_gravedad) + 1)
)

dim_gravedad["nivel_severidad"] = (
    dim_gravedad["gravedad_estandarizada"]
    .map({
        "NEGATIVO": 0,
        "SOLO DANOS": 1,
        "CON HERIDOS": 2,
        "CON FALLECIDOS": 4,
        "SIN DATO": 0,
        "OTRO": 0
    })
    .fillna(0)
    .astype(int)
)

dim_gravedad["es_siniestro_positivo"] = ~(
    dim_gravedad["gravedad_estandarizada"]
    .isin(["NEGATIVO", "SIN DATO"])
)

display(dim_gravedad)

,id_gravedad,tipo_confirmado_raw,gravedad_estandarizada,nivel_severidad,es_siniestro_positivo
0,1,CON FALLECIDO (FORANEO),CON FALLECIDOS,4,True
1,2,CON LESIONADO,CON HERIDOS,2,True
2,3,NEGATIVO,NEGATIVO,0,False
3,4,CON MUERTOS,OTRO,0,True
4,5,SOLO DAÑOS,SOLO DANOS,1,True


In [11]:
## Se crea la relación con la tabla de hechos 

mapa_gravedad = dict(
    zip(
        dim_gravedad["gravedad_estandarizada"],
        dim_gravedad["id_gravedad"]
    )
)

df["id_gravedad"] = df["gravedad_estandarizada"].map(mapa_gravedad)

# Dimensión Tipo Accidentes

In [12]:
dim_tipo_accidente = (
    df[
        [
            "tipo_accidente_raw",
            "tipo_accidente_estandarizado"
        ]
    ]
    .drop_duplicates("tipo_accidente_estandarizado")
    .sort_values("tipo_accidente_estandarizado")
    .reset_index(drop=True)
)

dim_tipo_accidente.insert(
    0,
    "id_tipo_accidente",
    range(1, len(dim_tipo_accidente) + 1)
)

display(dim_tipo_accidente)

,id_tipo_accidente,tipo_accidente_raw,tipo_accidente_estandarizado
0,1,ATROPELLO,ATROPELLO
1,2,CA?DA OCUPANTE,CAIDA DE OCUPANTE
2,3,CHOQUE,CHOQUE
3,4,INCENDIO,INCENDIO
4,5,OTRO,OTRO
5,6,<NA>,SIN DATO
6,7,VOLCAMIENTO,VOLCAMIENTO


In [13]:
## Se crea la relación con la tabla de hechos 
mapa_accidente = dict(
    zip(
        dim_tipo_accidente["tipo_accidente_estandarizado"],
        dim_tipo_accidente["id_tipo_accidente"]
    )
)

df["id_tipo_accidente"] = (
    df["tipo_accidente_estandarizado"]
    .map(mapa_accidente)
)

# Dimensión Medio Reporte

In [14]:
dim_medio_reporte = (
    df[
        [
            "medio_reporte",
            "medio_reporte_estandarizado"
        ]
    ]
    .drop_duplicates("medio_reporte_estandarizado")
    .sort_values("medio_reporte_estandarizado")
    .reset_index(drop=True)
)

dim_medio_reporte.insert(
    0,
    "id_medio_reporte",
    range(1, len(dim_medio_reporte) + 1)
)

dim_medio_reporte["es_reporte_agente"] = (
    dim_medio_reporte["medio_reporte_estandarizado"]
    == "AGENTE DE TRANSITO"
)

display(dim_medio_reporte)

,id_medio_reporte,medio_reporte,medio_reporte_estandarizado,es_reporte_agente
0,1,AGENTE TRACR,AGENTE DE TRANSITO,True
1,2,AGENTE CAD 123,LINEA 123,False
2,3,CAT 127,LINEA 127,False
3,4,MOVILIDAD CALI,OTRO-NARRATIVA,False
4,5,AGENTE RADIO,RADIO,False
5,6,REDES SOCIALES,REDES SOCIALES,False
6,7,<NA>,SIN DATO,False


In [15]:
mapa_medio = dict(
    zip(
        dim_medio_reporte["medio_reporte_estandarizado"],
        dim_medio_reporte["id_medio_reporte"]
    )
)

df["id_medio_reporte"] = (
    df["medio_reporte_estandarizado"]
    .map(mapa_medio)
)

# Dimensión Ubicación

In [16]:
dim_ubicacion = (
    df[["direccion_reporte"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_ubicacion["direccion_reporte"] = (
    dim_ubicacion["direccion_reporte"]
    .fillna("SIN DATO")
)

dim_ubicacion["tipo_via_principal"] = np.select(
    [
        dim_ubicacion["direccion_reporte"].str.startswith("CARRERA"),
        dim_ubicacion["direccion_reporte"].str.startswith("CALLE"),
        dim_ubicacion["direccion_reporte"].str.startswith("DIAGONAL"),
        dim_ubicacion["direccion_reporte"].str.startswith("TRANSVERSAL"),
        dim_ubicacion["direccion_reporte"].str.startswith("AVENIDA"),
        dim_ubicacion["direccion_reporte"].eq("SIN DATO")
    ],
    [
        "CARRERA",
        "CALLE",
        "DIAGONAL",
        "TRANSVERSAL",
        "AVENIDA",
        "SIN DATO"
    ],
    default="OTRO"
)

dim_ubicacion["es_interseccion"] = (
    dim_ubicacion["direccion_reporte"]
    .str.contains(r"\bCON\b", regex=True, na=False)
)

dim_ubicacion.insert(
    0,
    "id_ubicacion",
    range(1, len(dim_ubicacion) + 1)
)

display(dim_ubicacion.head())
print(f"Registros en DIM_UBICACION: {len(dim_ubicacion):,}")

,id_ubicacion,direccion_reporte,tipo_via_principal,es_interseccion
0,1,CALLE 36 & CARRERA 47,CALLE,False
1,2,CALLE 33 & CARRERA 98B,CALLE,False
2,3,CALLE 6 & CARRERA 27,CALLE,False
3,4,CALLE 16 & CARRERA 83,CALLE,False
4,5,CALLE 70 & CARRERA 5,CALLE,False


Registros en DIM_UBICACION: 59,360


In [17]:
mapa_ubicacion = dict(
    zip(
        dim_ubicacion["direccion_reporte"],
        dim_ubicacion["id_ubicacion"]
    )
)

df["id_ubicacion"] = (
    df["direccion_reporte"]
    .fillna("SIN DATO")
    .map(mapa_ubicacion)
)

# Dimensión Tipo Vehículo

In [18]:
## Se separan los tipos
vehiculos = (
    df[
        [
            "id_siniestro",
            "tipos_vehiculos"
        ]
    ]
    .dropna(subset=["tipos_vehiculos"])
    .assign(
        tipo_vehiculo_raw=lambda tabla: tabla["tipos_vehiculos"]
        .str.split(",")
    )
    .explode("tipo_vehiculo_raw")
)

vehiculos["tipo_vehiculo_raw"] = (
    vehiculos["tipo_vehiculo_raw"]
    .str.strip()
    .str.upper()
)

vehiculos = vehiculos[
    [
        "id_siniestro",
        "tipo_vehiculo_raw"
    ]
].drop_duplicates()

In [19]:
dim_tipo_vehiculo = (
    vehiculos[["tipo_vehiculo_raw"]]
    .drop_duplicates()
    .sort_values("tipo_vehiculo_raw")
    .reset_index(drop=True)
)

dim_tipo_vehiculo.insert(
    0,
    "id_tipo_vehiculo",
    range(1, len(dim_tipo_vehiculo) + 1)
)

texto_vehiculo = dim_tipo_vehiculo["tipo_vehiculo_raw"]

dim_tipo_vehiculo["tipo_vehiculo_estandarizado"] = (
    texto_vehiculo
    .str.replace("BUSPADRON", "BUS PADRON", regex=False)
)

dim_tipo_vehiculo["categoria_vehiculo"] = np.select(
    [
        texto_vehiculo.str.contains("MOTO", na=False),
        texto_vehiculo.str.contains("BICI", na=False),
        texto_vehiculo.str.contains(
            "CAMION|BUS|TRACTOCAMION",
            na=False
        ),
        texto_vehiculo.str.contains(
            "MAQUINARIA|TRACTOR",
            na=False
        ),
        texto_vehiculo.str.contains(
            "AUTOMOVIL|TAXI|CAMPERO",
            na=False
        )
    ],
    [
        "MOTOCICLETA",
        "NO MOTORIZADO",
        "PESADO",
        "MAQUINARIA",
        "LIVIANO"
    ],
    default="OTRO"
)

display(dim_tipo_vehiculo.head())

,id_tipo_vehiculo,tipo_vehiculo_raw,tipo_vehiculo_estandarizado,categoria_vehiculo
0,1,"""AMBULANCIA CAMPERO""","""AMBULANCIA CAMPERO""",LIVIANO
1,2,"""AUTOMOVIL TURISMO""","""AUTOMOVIL TURISMO""",LIVIANO
2,3,"""AUTOMOVILTURISMO""","""AUTOMOVILTURISMO""",LIVIANO
3,4,"""BUS ALIMENTADOR""","""BUS ALIMENTADOR""",PESADO
4,5,"""BUS ARTICULADO""","""BUS ARTICULADO""",PESADO


In [20]:
mapa_vehiculo = dict(
    zip(
        dim_tipo_vehiculo["tipo_vehiculo_raw"],
        dim_tipo_vehiculo["id_tipo_vehiculo"]
    )
)

bridge_siniestro_vehiculo = vehiculos.copy()

bridge_siniestro_vehiculo["id_tipo_vehiculo"] = (
    bridge_siniestro_vehiculo["tipo_vehiculo_raw"]
    .map(mapa_vehiculo)
)

cantidad_tipos = (
    bridge_siniestro_vehiculo
    .groupby("id_siniestro")["id_tipo_vehiculo"]
    .transform("nunique")
)

bridge_siniestro_vehiculo["peso"] = 1 / cantidad_tipos

bridge_siniestro_vehiculo = bridge_siniestro_vehiculo[
    [
        "id_siniestro",
        "id_tipo_vehiculo",
        "peso"
    ]
]

display(bridge_siniestro_vehiculo.head())

,id_siniestro,id_tipo_vehiculo,peso
370971,AI013516,21,0.5
370971,AI013516,42,0.5
370950,AI013495,42,0.5
370950,AI013495,25,0.5
370967,AI013512,42,0.5


# Puente Siniestro-Vehículo

In [21]:
mapa_vehiculo = dict(
    zip(
        dim_tipo_vehiculo["tipo_vehiculo_raw"],
        dim_tipo_vehiculo["id_tipo_vehiculo"]
    )
)

bridge_siniestro_vehiculo = vehiculos.copy()

bridge_siniestro_vehiculo["id_tipo_vehiculo"] = (
    bridge_siniestro_vehiculo["tipo_vehiculo_raw"]
    .map(mapa_vehiculo)
)

cantidad_tipos = (
    bridge_siniestro_vehiculo
    .groupby("id_siniestro")["id_tipo_vehiculo"]
    .transform("nunique")
)

bridge_siniestro_vehiculo["peso"] = 1 / cantidad_tipos

bridge_siniestro_vehiculo = bridge_siniestro_vehiculo[
    [
        "id_siniestro",
        "id_tipo_vehiculo",
        "peso"
    ]
]

display(bridge_siniestro_vehiculo.head())

,id_siniestro,id_tipo_vehiculo,peso
370971,AI013516,21,0.5
370971,AI013516,42,0.5
370950,AI013495,42,0.5
370950,AI013495,25,0.5
370967,AI013512,42,0.5


In [22]:
print("DIM_TIEMPO:", len(dim_tiempo))
print("DIM_GRAVEDAD:", len(dim_gravedad))
print("DIM_TIPO_ACCIDENTE:", len(dim_tipo_accidente))
print("DIM_UBICACION:", len(dim_ubicacion))
print("DIM_MEDIO_REPORTE:", len(dim_medio_reporte))
print("DIM_TIPO_VEHICULO:", len(dim_tipo_vehiculo))
print("BRIDGE:", len(bridge_siniestro_vehiculo))

print("\nClaves faltantes:")
print("id_tiempo:", df["fecha"].isna().sum())
print("id_gravedad:", df["id_gravedad"].isna().sum())
print("id_tipo_accidente:", df["id_tipo_accidente"].isna().sum())
print("id_ubicacion:", df["id_ubicacion"].isna().sum())
print("id_medio_reporte:", df["id_medio_reporte"].isna().sum())

DIM_TIEMPO: 3229
DIM_GRAVEDAD: 5
DIM_TIPO_ACCIDENTE: 7
DIM_UBICACION: 59360
DIM_MEDIO_REPORTE: 7
DIM_TIPO_VEHICULO: 54
BRIDGE: 219711

Claves faltantes:
id_tiempo: 0
id_gravedad: 0
id_tipo_accidente: 0
id_ubicacion: 0
id_medio_reporte: 0


# Crear la tabla de hechos Fact_Siniestro

In [23]:
df["id_tiempo"] = (
    df["fecha"]
    .dt.strftime("%Y%m%d")
    .astype("Int64")
)

In [24]:
fact_siniestro = df[
    [
        "id_siniestro",
        "id_tiempo",
        "id_gravedad",
        "id_tipo_accidente",
        "id_ubicacion",
        "id_medio_reporte",
        "numero_vehiculos"
    ]
].copy()

fact_siniestro["cantidad_siniestros"] = 1

display(fact_siniestro.head())
print(f"Registros en FACT_SINIESTRO: {len(fact_siniestro):,}")

,id_siniestro,id_tiempo,id_gravedad,id_tipo_accidente,id_ubicacion,id_medio_reporte,numero_vehiculos,cantidad_siniestros
370971,AI013516,20241102,2,3,1,5,2,1
370950,AI013495,20241101,2,3,2,3,2,1
370967,AI013512,20241102,3,3,3,5,3,1
370968,AI013513,20241102,2,3,4,5,2,1
370941,AI013486,20241101,3,3,5,3,1,1


Registros en FACT_SINIESTRO: 370,998


In [25]:
print("Filas del DataFrame original:", len(df))
print("Filas de FACT_SINIESTRO:", len(fact_siniestro))
print("Siniestros duplicados:", fact_siniestro["id_siniestro"].duplicated().sum())

print("\nClaves faltantes en la tabla de hechos:")
display(fact_siniestro.isna().sum())

Filas del DataFrame original: 370998
Filas de FACT_SINIESTRO: 370998
Siniestros duplicados: 0

Claves faltantes en la tabla de hechos:


id_siniestro                0
id_tiempo                   0
id_gravedad                 0
id_tipo_accidente           0
id_ubicacion                0
id_medio_reporte            0
numero_vehiculos       194421
cantidad_siniestros         0
dtype: int64

# Conexión PostgreSQL

In [26]:
%pip install pandas sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [27]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError(
        "No se encontró DATABASE_URL en el archivo .env."
    )

try:
    engine = create_engine(DATABASE_URL)

    with engine.connect() as conexion:
        conexion.execute(text("SELECT 1"))

    print("Conexión exitosa a PostgreSQL.")

except SQLAlchemyError as error:
    raise ConnectionError(
        "No fue posible conectarse a PostgreSQL. "
        "Verifica que el servidor esté activo, que la base de datos exista "
        "y que DATABASE_URL sea correcta."
    ) from error

Conexión exitosa a PostgreSQL.


# Carga de los DataFrames

In [28]:
tablas_carga = {
    "dim_tiempo": dim_tiempo,
    "dim_gravedad": dim_gravedad,
    "dim_tipo_accidente": dim_tipo_accidente,
    "dim_ubicacion": dim_ubicacion,
    "dim_medio_reporte": dim_medio_reporte,
    "dim_tipo_vehiculo": dim_tipo_vehiculo,
    "fact_siniestro": fact_siniestro,
    "bridge_siniestro_vehiculo": bridge_siniestro_vehiculo
}

try:
    for nombre_tabla, tabla in tablas_carga.items():
        tabla.to_sql(
            nombre_tabla,
            con=engine,
            if_exists="replace",
            index=False,
            method="multi",
            chunksize=5000
        )

        print(f"Tabla cargada correctamente: {nombre_tabla}")

except SQLAlchemyError as error:
    raise RuntimeError(
        f"Error durante la carga de datos: {error}"
    ) from error

Tabla cargada correctamente: dim_tiempo
Tabla cargada correctamente: dim_gravedad
Tabla cargada correctamente: dim_tipo_accidente
Tabla cargada correctamente: dim_ubicacion
Tabla cargada correctamente: dim_medio_reporte
Tabla cargada correctamente: dim_tipo_vehiculo
Tabla cargada correctamente: fact_siniestro
Tabla cargada correctamente: bridge_siniestro_vehiculo


In [29]:
## Verificar si cargaron los documentos

with engine.connect() as conexion:
    for nombre_tabla in tablas_carga:
        cantidad = conexion.execute(
            text(f'SELECT COUNT(*) FROM "{nombre_tabla}"')
        ).scalar_one()

        print(f"{nombre_tabla}: {cantidad:,} registros")

dim_tiempo: 3,229 registros
dim_gravedad: 5 registros
dim_tipo_accidente: 7 registros
dim_ubicacion: 59,360 registros
dim_medio_reporte: 7 registros
dim_tipo_vehiculo: 54 registros
fact_siniestro: 370,998 registros
bridge_siniestro_vehiculo: 219,711 registros


In [30]:
with engine.connect() as conexion:
    resultado = conexion.execute(
        text("SELECT current_database(), current_user")
    ).fetchone()

print(f"Base de datos utilizada: {resultado[0]}")
print(f"Usuario utilizado: {resultado[1]}")
print("Proceso ETL finalizado correctamente.")

Base de datos utilizada: siniestralidad_cali
Usuario utilizado: postgres
Proceso ETL finalizado correctamente.
